# TIER 0 — Gate experiments 
(days 1–3, everything downstream depends on these)

## E1. Stratified residue test (n=14, 509 graphs, full vectors).
Fix triangle count, rerun Mantel/PCA within each stratum (tri=0 and tri=2 bands are largest). Then fix tri+C4, look for C5.
Proves: the embedding is layered, not a triangle counter — the accessibility-hierarchy claim.
Branch: C4 organizes within strata → hierarchy paper, proceed to full plan. It doesn't → fall back to "parameter-robust quantum triangle estimator + truncation study"; cut E5/E6, ship E2/E3/E7, consider whether that's AAAI-worthy or a quantum-venue paper. Decide by day 3, not later.

## Configuration

In [1]:
N_VERTICES = 14           # 10 -> 19 graphs, 12 -> 85, 14 -> 509, 16 -> 4060
MAX_CYCLE_LEN = 5          # count cycles of length 3..MAX_CYCLE_LEN

# Circuit parameters for this run.
MAX_ENCODING_ROTATION = 2.875
ENTANGLER_SCALAR = 2.0
MIXER_SCALAR = 0.1
ENCODING_SCALAR = None     # None -> MAX_ENCODING_ROTATION / max_degree
REPS = 1  # We really want to try to do this paper with r=1.  We then get to fully claim injectivity.

# We're in the simulable regime, and $\leq 2^14$ is manageable. r=2 is a minor improvement on the most difficult cases.
TRUNCATION_LENGTH = None    # We're in the simulable regime, and $\leq 2^14$ is manageable
SEED = 0                   # controls Mantel permutations and the random control
N_PERM = 10_000

## Environment

In [2]:
!pip install 'qiskit[visualization]' --quiet
!apt-get install -y nauty > /dev/null && which nauty-geng

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 85.9 MB/s eta 0:00:00
/usr/bin/nauty-geng


In [3]:
import subprocess
import pickle

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit as QC
from qiskit.quantum_info import Statevector

from scipy.stats import spearmanr, rankdata
from scipy.spatial.distance import pdist, squareform

from tqdm.notebook import tqdm

## Graph generation — exhaustive connected cubic graphs

`nauty-geng -c -d3 -D3 n` enumerates every connected 3-regular graph on n vertices,
one graph6 string per line. Known counts are asserted so a broken install or a
changed flag fails loudly instead of silently biasing the study.

In [4]:
EXPECTED_CUBIC_COUNTS = {10: 19, 12: 85, 14: 509, 16: 4060}

def cubic_graphs(n):
    out = subprocess.run(['nauty-geng', '-c', '-d3', '-D3', str(n)],
                         capture_output=True, text=True, check=True)
    g6_strings = out.stdout.split()
    graphs = [nx.from_graph6_bytes(s.encode()) for s in g6_strings]
    if n in EXPECTED_CUBIC_COUNTS:
        assert len(graphs) == EXPECTED_CUBIC_COUNTS[n], (
            f'expected {EXPECTED_CUBIC_COUNTS[n]} graphs at n={n}, got {len(graphs)}')
    return graphs, g6_strings

graph_list, g6_list = cubic_graphs(N_VERTICES)
print(f'{len(graph_list)} connected cubic graphs at n={N_VERTICES}')

509 connected cubic graphs at n=14


## Cycle counts via networkx

`nx.simple_cycles(G, length_bound=k)` (networkx >= 3.1, undirected support)
yields each simple cycle exactly once — no closed-walk corrections, no
double-counting algebra. Validated against known graphs before use.

In [5]:
def cycle_counts(graph, max_len=MAX_CYCLE_LEN):
    """Counts of simple cycles by length, 3..max_len. Each cycle counted once."""
    counts = {k: 0 for k in range(3, max_len + 1)}
    for cyc in nx.simple_cycles(graph, length_bound=max_len):
        counts[len(cyc)] += 1
    return counts

## QuIC circuit

In [6]:
class QuIK_circuit:
    """Builds the QuIC circuit for a given adjacency matrix.

    Degree-proportional RX encoder, RZZ entangler over edges, uniform RX mixer,
    (entangler + mixer) repeated `reps` times. Construction only — sampling,
    sorting, and truncation happen downstream.
    """

    def __init__(self, adj_mat,
                 max_encoding_rotation=MAX_ENCODING_ROTATION,
                 entangler_scalar=ENTANGLER_SCALAR,
                 mixer_scalar=MIXER_SCALAR,
                 encoding_scalar=ENCODING_SCALAR,
                 degree_sequence=None,
                 reps=REPS):
        self.adj_mat = np.asarray(adj_mat)

        if degree_sequence is None:
            self.degree_sequence = self.adj_mat.sum(axis=1)
        else:
            if not np.array_equal(self.adj_mat.sum(axis=1), degree_sequence):
                raise ValueError('provided degree_sequence does not match adj_mat')
            self.degree_sequence = degree_sequence
        self.max_degree = self.degree_sequence.max()

        self.max_encoding_rotation = max_encoding_rotation
        self.entangler_scalar = entangler_scalar
        self.mixer_scalar = mixer_scalar
        self.reps = reps

        self.num_qubits = self.adj_mat.shape[0]
        self.qubits = list(range(self.num_qubits))
        self.encoding_scalar = (self.max_encoding_rotation / self.max_degree
                                if encoding_scalar is None else encoding_scalar)

        self.quik_circuit = self.build_quik_circuit()

    def build_encoder(self):
        encoder = QC(self.num_qubits)
        for i, degree in enumerate(self.degree_sequence):
            encoder.rx(self.encoding_scalar * degree, i, 'Degree Encoder')
        return encoder

    def build_entangler(self):
        entangler = QC(self.num_qubits)
        edge_u, edge_v = np.nonzero(np.triu(self.adj_mat, k=1))
        for u, v in zip(edge_u, edge_v):
            entangler.rzz(self.entangler_scalar, u, v)
        return entangler

    def build_mixer(self):
        mixer = QC(self.num_qubits)
        mixer.rx(self.mixer_scalar, self.qubits, 'Mixer')
        return mixer

    def build_quik_circuit(self):
        self.encoder = self.build_encoder()
        self.mixer = self.build_mixer()
        self.entangler = self.build_entangler()

        qc = self.encoder.copy()
        for _ in range(self.reps):
            qc.append(self.entangler, self.qubits)
            qc.append(self.mixer, self.qubits)
        qc.measure_all()
        return qc

## Data assembly

One `data_dict` keyed by graph index; graph6 string retained for provenance.

In [7]:
data_dict = {}
for i, (graph, g6) in enumerate(zip(graph_list, g6_list)):
    cc = cycle_counts(graph)
    data_dict[i] = {
        'graph': graph,
        'graph6': g6,
        'adj_mat': nx.to_numpy_array(graph),
        'circuit': QuIK_circuit(nx.to_numpy_array(graph)).quik_circuit.decompose(),
        'num_triangles': cc[3],
        'num_4_cycles': cc[4],
        'num_5_cycles': cc[5],
        'exact_vector': None,
    }
print(f'{len(data_dict)} graphs assembled')

509 graphs assembled


In [8]:
tri_max = 0
quad_max = 0
pent_max = 0
for value in data_dict.values():
    tri_max = max(tri_max, value["num_triangles"])
    quad_max = max(quad_max, value["num_4_cycles"])
    pent_max = max(pent_max, value["num_5_cycles"])
tri_max, quad_max, pent_max

(6, 10, 10)

In [9]:
# Replace hardcoded label ranges with observed maxima — a tri_7 graph at n=14
# KeyErrors the assembly under the current 0..6 range.
tri_labels  = [f"tri_{x}"  for x in range(tri_max + 1)]
quad_labels = [f"quad_{x}" for x in range(quad_max + 1)]
pent_labels = [f"pent_{x}" for x in range(pent_max + 1)]
all_labels = tri_labels + quad_labels + pent_labels
print(all_labels)


['tri_0', 'tri_1', 'tri_2', 'tri_3', 'tri_4', 'tri_5', 'tri_6', 'quad_0', 'quad_1', 'quad_2', 'quad_3', 'quad_4', 'quad_5', 'quad_6', 'quad_7', 'quad_8', 'quad_9', 'quad_10', 'pent_0', 'pent_1', 'pent_2', 'pent_3', 'pent_4', 'pent_5', 'pent_6', 'pent_7', 'pent_8', 'pent_9', 'pent_10']


### Explore the space of these graphs

In [10]:
structure_key_dict = {x: [] for x in all_labels}
for key, value in data_dict.items():
    tri_val = value["num_triangles"]
    quad_val = value["num_4_cycles"]
    pent_val = value["num_5_cycles"]
    structure_key_dict[f"tri_{tri_val}"].append(key)
    structure_key_dict[f"quad_{quad_val}"].append(key)
    structure_key_dict[f"pent_{pent_val}"].append(key)

In [11]:
for _, graph_dict in tqdm(data_dict.items()):
    circuit = graph_dict['circuit'].remove_final_measurements(inplace=False)
    probs = Statevector.from_instruction(circuit).probabilities()
    graph_dict['exact_vector'] = np.sort(probs)[::-1][:]

  0%|          | 0/509 [00:00<?, ?it/s]

In [12]:
# --- Extra targets (cheap now, expensive to want later) ---
from numpy.linalg import eigvalsh

for k in data_dict.keys():
    A = data_dict[k]['adj_mat']
    spec = np.sort(eigvalsh(A))
    data_dict[k]['spectrum'] = spec
    data_dict[k]['spectral_gap'] = spec[-1] - spec[-2]
    data_dict[k]['girth'] = nx.girth(data_dict[k]['graph'])
    data_dict[k]['diameter'] = nx.diameter(data_dict[k]['graph'])



# # Cospectral mates: zero spectral distance, any embedding separation is
# # super-spectral content by construction. Free exhibit if any exist at n=14.
# spec_hash = [tuple(np.round(s, 8)) for s in spectra]
# from collections import defaultdict
# mates = defaultdict(list)
# for k, h in zip(keys, spec_hash):
#     mates[h].append(k)
# cospectral_groups = [v for v in mates.values() if len(v) > 1]

In [13]:
readme = """QuIC EXACT EMBEDDINGS — EXHAUSTIVE CONNECTED CUBIC GRAPHS, n=14
=================================================================

Generated: 2026-07-09
Producer:  quic-cycle-correlation-n14 (Kaggle)
Author:    Luke Miller (UMKC)
Context:   Decodability study of the QuIC quantum graph embedding
           (companion to QCE 2026 paper, arXiv:2604.18841).

CONTENTS
--------
n14_data_dict.pkl
    dict keyed by graph index (0..508), one entry per graph in the
    complete enumeration of connected 3-regular graphs on 14 vertices
    (509 graphs, generated by nauty-geng -c -d3 -D3 14, order as emitted).
    Per-entry fields:
      graph6         : str   — canonical graph6 string (reconstruct graph via
                               nx.from_graph6_bytes(s.encode()); circuit via
                               QuIK_circuit(adj_mat) with config below)
      adj_mat        : (14,14) float array — adjacency matrix
      num_triangles  : int   — simple 3-cycles (nx.simple_cycles, each once)
      num_4_cycles   : int   — simple 4-cycles
      num_5_cycles   : int   — simple 5-cycles
      exact_vector   : (16384,) array — QuIC embedding: exact statevector
                       output probabilities, sorted non-increasing, FULL
                       vector (no truncation)
    networkx graph objects and qiskit circuit objects are deliberately
    excluded (version-fragile in pickle; fully reconstructable).

n14_structure_key_dict.pkl
    dict: stratum label -> list of graph indices.
    Labels: 'tri_{v}', 'quad_{v}', 'pent_{v}' for observed count values v.
    Index for stratified analyses (fix one count, test within stratum).

CIRCUIT CONFIG (canonical angles, one repetition)
-------------------------------------------------
    max_encoding_rotation = 2.875     # theta_enc
    entangler_scalar      = 2.0       # theta_ent
    mixer_scalar          = 0.1       # theta_mix
    encoding_scalar       = None      # -> 2.875 / max_degree = 2.875/3
    reps                  = 1         # repetition regime covered by the
                                      # injectivity theorem (QCE Thm 3)
    embedding             = exact statevector, noiseless, exact arithmetic
    truncation            = NONE (full 2^14 sorted vector)
    seed                  = 0

NOTES
-----
- Cubic graphs: degree encoder is uniform by construction; all
  discriminative signal originates in the RZZ entangling layer.
- Sorted probabilities only: bitstring identities are discarded
  (permutation invariance by sorting; see QCE paper Sec. III).
- Counts computed with networkx >= 3.1 simple_cycles(length_bound=5);
  validated against K4, C5, Petersen.
- Recommended cast for analysis at this scale: float32 is sufficient;
  vectors are stored as produced.
- Known-count assertion at generation: 509 graphs (Meringer/OEIS A002851).

CHANGELOG
---------
v1 (2026-07-09): initial generation.
"""

# Strip version-fragile objects before dumping — graph6 + config block
# reconstructs both. (Do NOT reuse data_dict for in-session work after this
# cell without rebuilding those fields.)
for entry in data_dict.values():
    entry.pop('graph', None)
    entry.pop('circuit', None)

with open("/kaggle/working/n14_data_dict.pkl", "wb") as f:
    pickle.dump(data_dict, f)
with open("/kaggle/working/n14_structure_key_dict.pkl", "wb") as f:
    pickle.dump(structure_key_dict, f)
with open("/kaggle/working/n14_readme.pkl", "wb") as f:
    pickle.dump(readme, f)

# Plain-text copy so the Kaggle dataset preview shows it without unpickling:
with open("/kaggle/working/README.txt", "w") as f:
    f.write(readme)

print(f"dumped {len(data_dict)} entries; "
      f"vector dtype {data_dict[0]['exact_vector'].dtype}, "
      f"dim {data_dict[0]['exact_vector'].shape[0]}")

dumped 509 entries; vector dtype float64, dim 16384
